In [ ]:
import torch
from torchvision import datasets, models, transforms

# Models ja existents i *transfer learning*

En aquesta pràctica, aprofundirem en la classificació d’imatges amb xarxes neuronals convolucionals (CNNs), però amb un enfocament diferent al de la sessió anterior. Mentre que anteriorment vàrem construir CNNs des de zero per comprendre la seva estructura bàsica, aquest cop treballarem amb un model de CNN preentrenat: **AlexNet**.

### AlexNet


**AlexNet** és l'arquitectura de xarxa neuronal convolucional (CNN) dissenyada per Alex Krizhevsky, Ilya Sutskever i Geoffrey Hinton (Universitat de Toronto) que va transformar el camp de la visió per computador l'any 2012. En guanyar la competició **ILSVRC-2012 (ImageNet Large Scale Visual Recognition Challenge)** amb un error *top-5* del **15,3%** —enfront del 26,2% obtingut pel segon classificat, basat en mètodes clàssics—, va marcar el punt d'inflexió decisiu cap a l'ús massiu de xarxes neuronals profundes.

![Arquitectura](https://viso.ai/wp-content/uploads/2024/04/alexNet-architecture.png)

#### 1. Context i Impacte Històric

Abans d'AlexNet, la visió per computador depenia principalment de mètodes en dues etapes:
* **Extracció artesanal de característiques (*hand-crafted features*):** Algoritmes com SIFT, SURF o HOG dissenyats manualment per experts.
* **Classificadors superficials:** Models com Support Vector Machines (SVM) o Random Forests alimentats amb aquests descriptors.

AlexNet va demostrar empíricament la superioritat d'un nou paradigma:
1. **Aprenentatge extrem a extrem (*end-to-end*):** Els filtres i les jerarquies visuals s'aprenen automàticament des dels píxels bruts.
2. **Acceleració massiva en GPU:** L'ús de maquinari paral·lel va permetre entrenar milions de paràmetres en dies en lloc de setmanes o mesos.
3. **Escala massiva de dades:** L'ús de la base de dades ImageNet (més d'1,2 milions d'imatges d'entrenament repartides en 1.000 classes) va permetre generalitzar models amb gran capacitat sense sobreajustament immediat.

#### 2. Arquitectura de la Xarxa

AlexNet conté aproximadament **60 milions de paràmetres** i **650.000 neurones**, distribuïts al llarg de **8 capes d'aprenentatge amb pesos**: 5 capes convolucionals i 3 capes completament connectades (*fully connected* o FC).

| Capa | Tipus | Filtres / Mida del nucli (*Kernel*) | Pas (*Stride*) | Mida del mapa de sortida |
| :--- | :--- | :--- | :--- | :--- |
| **Entrada** | Imatge RGB | — | — | $224 \times 224 \times 3$ |
| **Conv 1** | Convolució + MaxPool + LRN | 96 filtres d'$11 \times 11$ | 4 | $55 \times 55 \times 96 \rightarrow 27 \times 27 \times 96$ |
| **Conv 2** | Convolució + MaxPool + LRN | 256 filtres de $5 \times 5$ | 1 (pad 2) | $27 \times 27 \times 256 \rightarrow 13 \times 13 \times 256$ |
| **Conv 3** | Convolució | 384 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 384$ |
| **Conv 4** | Convolució | 384 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 384$ |
| **Conv 5** | Convolució + MaxPool | 256 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 256 \rightarrow 6 \times 6 \times 256$ |
| **FC 6** | Densa + Dropout ($p=0.5$) | 4.096 neurones | — | 4.096 |
| **FC 7** | Densa + Dropout ($p=0.5$) | 4.096 neurones | — | 4.096 |
| **FC 8** | Densa (Sortida) | 1.000 classes (Softmax) | — | 1.000 |

---

### Transfer Learning?

El **Transfer Learning** és una tècnica d'aprenentatge profund que permet aprofitar xarxes neuronals prèviament entrenades en conjunts de dades massius (com ara **AlexNet** entrenada sobre ImageNet) per resoldre una tasca nova o específica.

En lloc d'iniciar l'entrenament des de zero amb pesos aleatoris —procés que requereix grans volums de dades i un elevat cost computacional—, adaptem l'arquitectura a les noves classes del nostre conjunt de dades. D'aquesta manera, el nou model es beneficia directament de les representacions jeràrquiques ja apreses per la xarxa base (patrons visuals universals com ara vores, textures, gradients o formes geomètriques bàsiques).

## Estratègies d'Adaptació

Segons el volum de dades disponible i la similitud entre la tasca d'origen i la nova tasca, aquest procés es pot dur a terme principalment mitjançant dues estratègies:

#### 1. *Transfer Learning* com a Extractor de Característiques Fix (*Feature Extraction*)

En aquest enfocament, s'assumeix que les característiques apreses per les capes convolucionals són suficients per a la nova tasca:

* **Congelació de l'extractor:** Es fixen els pesos de les capes convolucionals (`features`) desactivant el càlcul de gradients. D'aquesta manera, els pesos originals no es modifiquen durant la retropropagació.
* **Nou classificador:** Se substitueix la capa de sortida original (com ara el bloc fully-connected o `MLP`) per un nou capçal adaptat al nombre de classes de la nova tasca, el qual s'entrena completament des de zero.

```python
# Congelació de tots els paràmetres de l'extractor de característiques
for param in alex.features.parameters():
    param.requires_grad = False

# Adaptació del classificador per a la nova tasca (p. ex., 10 classes)
import torch.nn as nn
num_ftrs = alex.classifier[6].in_features
alex.classifier[6] = nn.Linear(num_ftrs, 10)
```

---

#### 2. *Fine-Tuning*

El **fine-tuning** consisteix a permetre que part de la xarxa preentrenada continuï aprenent per adaptar-se millor a les particularitats del nou domini:

* **Congelació parcial:** Es mantenen fixes les primeres capes del model (on es capturen els patrons més generals com línies i colors), però es descongelen les darreres capes convolucionals, més especialitzades en formes d'alt nivell.
* **Reentrenament conjunt:** Es reentrenen tant el nou classificador com el subconjunt seleccionat de capes convolucionals.
* **Taxes d'aprenentatge diferenciades (*Differential Learning Rates*):** Per evitar destruir el coneixement previ (*catastrophic forgetting*), s'utilitza una taxa d'aprenentatge (*learning rate*) substancialment **més petita** per a les capes preentrenades que per al nou classificador, utilitzant grups de paràmetres a l'optimitzador:

```python
import torch.optim as optim

# Configuració de taxes d'aprenentatge diferenciades per grup de capes
optimizer = optim.Adam([
    {"params": model.features[8:].parameters(), "lr": 1e-5},  # Capes convolucionals profundes
    {"params": model.classifier.parameters(),   "lr": 1e-3},  # Classificador nou
])
```

## Començam

Primer de tot, com sempre, hem d'obtenir les dades. Aquesta sessió la farem amb el conjunt de dades [Tiny ImageNet](https://www.kaggle.com/c/tiny-imagenet/data?select=train.images.zip).

Aquest conjunt de dades es defineix en la seva plana de la forma següent:

> MicroImageNet classification challenge is similar to the classification challenge in the full ImageNet ILSVRC. MicroImageNet contains 200 classes for training. Each class has 500 images. The test set contains 10,000 images. All images are 64x64 colored ones.

Aquesta vegada no farem feina amb un conjunt de dades ja existents a ``torchvision`` sinó que nosaltres farem la gestió des de 0. Per tant i primer de tot descarregarem les dades. Per fer-ho podem trobar el conjunt de dades a la següent plana (http://cs231n.stanford.edu/tiny-imagenet-200.zip).

Alternativament també podem emprar l'eina ``wget`` així:

```
wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
```

Una vegada que hem descarregat les dades les podem descomprimir i finalment comença a fer-hi feina.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


train = datasets.ImageFolder('../../data/tiny-imagenet/train', transform=transform)
test = datasets.ImageFolder('../../data/tiny-imagenet/val', transform=transform)

train_loader = torch.utils.data.DataLoader(train,
                                           batch_size=BATCH_SIZE,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(test,
                                          batch_size=BATCH_SIZE,
                                          shuffle=True)

In [ ]:
img, target = next(iter(train_loader))
print(img.shape, target)

## Definició de la xarxa: AlexNet i *Transfer learning*

En aquesta pràctica aplicarem la tècnica de transfer learning amb la primera xarxa CNN moderna:
- AlexNet. [ImageNet Classification with Deep Convolutional Neural Network, 2012](https://proceedings.neurips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf). La mida d'entrada de les imatges és de (227x227x3).Té prop de 60 milions de paràmetres entrenables.

Pytorch ens permet emprar aquest tipus de xarxes de manera molt senzilla. [Més informació](https://pytorch.org/vision/stable/models.html). Si el model que cercam no es troba integrat dins la llibreria Pytorch és bastant probable que si la trobem a Huggingface.

Descarregarem AlexNet i a analitzar-la. En aquest cas no només ens baixam la seva arquitectura, també els pesos resultants de l'entrenament.

**Normalment els problems els resoldrem emprant models ja definits i preentrenats**

In [ ]:
alex = models.alexnet(weights=True)

print("-" * 50)
print("Arquitectura AlexNet")
print("-" * 50)
print(alex)

Podem accedir a una capa concreta pel seu nom o índex dins de l’estructura del model. Per exemple:

In [ ]:
alex.features[0]

El que nosaltres volem fer és emprar els pesos ja entrenats d'aquest model i aplicar-ho per resoldre un problema nou. Hi ha diverses maneres de realitzar la tècnica de ``Transfer Learning``. Les dues les hem explicades a la introducció, en aquest cas però emprarem el ``transfer learning`` pròpiament dit. L'element principal és congelar les capes de l'extractor de característiques. Per fer-ho empram les següents instruccions

```
for param in alex.features.parameters():
   param.requires_grad = False

```

## Feina a fer:

1. Carregar la xarxa AlexNet:
   1. Emprar el model sense reentrenar.
   2. Entrenar el model emprant *fine-tunning*.
   3. Entrenar el model emprant *transfer learning*.
2. Avaluar els diferents models.
3. Guardau els millors pesos de les vostres xarxes i tornau-los a carregar. Classificar una imatge del conjunt de test.

**Nota**. Com veureu no us donam aquesta vegada el bucle d'entrenament, sigui com sigui podeu adaptar el vist a les sessions anteriors.


## EXTRA: Com emprar la GPU per entrenar un model

Un dels elements diferencials d'aquest model, respecte als que havíem vist fins ara, és la seva mida i, per tant, l'entrenament es torna impossible emprant __CPU__ directament. Per resoldre-ho hem d'emprar una **GPU**, a Google Colab disposam d'elles gratuïtament. Per fer-ho amb *Pytorch* hem de fer tres passes:

1. Comprovar que hi ha una GPU disponible.
2. Moure el model a GPU.
3. Moure les dades a GPU.

### Comprova si tenim una GPU disponible

Primer de tot, cal verificar si hi ha una GPU disponible a l’entorn. Això es pot fer amb el següent codi:

```python

import torch

is_cuda = torch.cuda.is_available()
```

Si la variable ``is_cuda`` és certa, llavors tens accés a una GPU.

### Mou el model a la GPU

En PyTorch, els models han d'estar explícitament en la GPU per poder fer servir la seva potència de càlcul. Si estàs carregant un model preentrenat (com AlexNet, ResNet, etc.), o si has definit el teu propi model, pots moure’l a la GPU amb ``.to(device)``, on device fa referència a la GPU.

```python

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
```

Això mou el model a la GPU (si està disponible). Si només tens una CPU, el model es mantindrà a la CPU.

### Mou les dades a la GPU

No només el model, sinó que també les dades (inputs) han d'estar a la GPU per fer les operacions més ràpides. Així, abans de fer servir les dades com a inputs del model, assegura't de moure-les al mateix device:

```python

# Exemple d'un batch de dades
inputs, labels = inputs.to(device), labels.to(device)
```

hola buenas